# 02 — Embedding Model Experiments

Compare all-mpnet-base-v2 vs all-MiniLM-L6-v2 on clustering quality.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from pathlib import Path

from featrank.ingest.csv_connector import CSVConnector
from featrank.pipeline.cleaner import TextCleaner
from featrank.pipeline.embedder import Embedder
from featrank.pipeline.clusterer import HDBSCANClusterer

In [ ]:
requests = CSVConnector('../tests/fixtures/sample_requests.csv').fetch_all()
cleaner = TextCleaner()
kept, cleaned_texts = cleaner.clean_requests(requests)
print(f'{len(kept)} requests after cleaning')

In [ ]:
MODELS = [
    'all-MiniLM-L6-v2',
    'all-mpnet-base-v2',
]

results = []
for model_name in MODELS:
    embedder = Embedder(model_name=model_name, cache_dir=f'.cache/{model_name}')
    t0 = time.perf_counter()
    embs = embedder.embed(cleaned_texts)
    elapsed = time.perf_counter() - t0

    clusterer = HDBSCANClusterer(min_cluster_size=5, min_samples=3)
    clustered = clusterer.fit_predict(kept, embs, cleaned_texts)
    labels = [r.cluster_id for r in clustered]
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = labels.count(-1)

    results.append({
        'model': model_name,
        'dim': embs.shape[1],
        'embed_time_s': round(elapsed, 2),
        'n_clusters': n_clusters,
        'n_noise': n_noise,
        'noise_pct': round(n_noise / len(labels) * 100, 1),
    })

df_results = pd.DataFrame(results)
df_results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

df_results.plot(x='model', y='embed_time_s', kind='bar', ax=axes[0], legend=False, title='Embedding Time (s)')
df_results.plot(x='model', y='n_clusters', kind='bar', ax=axes[1], legend=False, title='Clusters Found')
df_results.plot(x='model', y='noise_pct', kind='bar', ax=axes[2], legend=False, title='Noise %')

for ax in axes:
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()
print('\nConclusion: all-mpnet-base-v2 gives richer clusters at the cost of ~2x embedding time.')